# Notebook 05: Semantic Classification Model Training

**Phase:** Semantic Classification (Thesis Plan Deliverable 1)

**Purpose:** Train a multi-label classifier for semantic ingredient categorization using the labeled data from Notebook 04.

## Overview

Trains a MobileBERT-based multi-label classifier for semantic ingredient categories.
Reuses the same architecture and patterns from the allergen detection model (Notebook 07)
but with a different label space and task.

## Workflow

1. Load semantic labeling data from Notebook 04
2. Prepare multi-label dataset (ingredient text → semantic category vector)
3. Stratified train/val/test split
4. Train MobileBERT with weighted BCE loss
5. Evaluate and optimize per-class thresholds
6. Save model and evaluation metrics

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score

from utils.data_utils import load_labeled_data, create_stratified_splits, get_data_directories, save_metadata
from utils.semantic_utils import get_semantic_category_list, get_category_groups
from utils.model_utils import load_model_and_tokenizer, compute_class_weights, find_best_thresholds, predict_ml
from utils.evaluation import print_classification_report, error_analysis

# Set seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports completed.")

Imports completed.


## 1. Load Semantic Label Data

In [2]:
dirs = get_data_directories()
categories = get_semantic_category_list()
print(f"Semantic categories ({len(categories)}): {categories}")

# Load the ingredient-level training data
training_path = os.path.join(dirs['final'], 'semantic_training_data.csv')
data = pd.read_csv(training_path)

# Convert label vectors from strings back to lists
import ast
data['label_vector'] = data['label_vector'].apply(ast.literal_eval)

# Pad label vectors if the category list grew after the data was saved
n_loaded = len(data['label_vector'].iloc[0])
n_expected = len(categories)
if n_loaded < n_expected:
    pad_width = n_expected - n_loaded
    data['label_vector'] = data['label_vector'].apply(
        lambda v: v + [0] * pad_width
    )
    print(f"Padded label vectors: {n_loaded} → {n_expected} dimensions "
          f"(+{pad_width} new categories: {categories[n_loaded:]})")
elif n_loaded > n_expected:
    print(f"WARNING: Loaded {n_loaded}-dim vectors but expected {n_expected}. "
          "Truncating — data may be misaligned with current category list.")
    data['label_vector'] = data['label_vector'].apply(
        lambda v: v[:n_expected]
    )

print(f"\nTraining data shape: {data.shape}")
print(f"Products represented: {data['product_code'].nunique()}")
print(f"Unique ingredients: {data['ingredient'].nunique()}")
print(f"\nFirst 5 rows:")
data[['ingredient', 'semantic_labels']].head()

Semantic categories (43): ['food_additive', 'preservative', 'flavor_enhancer', 'sweetener', 'emulsifier', 'stabilizer', 'thickener', 'antioxidant', 'acidulant', 'colorant', 'fat_source', 'oil_source', 'protein_source', 'carbohydrate_source', 'animal_derived', 'plant_derived', 'milk_derivative', 'egg_derivative', 'soy_derivative', 'wheat_derivative', 'sodium_compound', 'sugar', 'added_sugar', 'fiber', 'vitamin', 'mineral', 'salt', 'yeast', 'leavening_agent', 'gelling_agent', 'humectant', 'flavoring', 'spice', 'herb', 'fruit_derived', 'vegetable_derived', 'fermented', 'smoked', 'cured', 'enzyme', 'culture', 'prebiotic', 'salt_substitute']

Training data shape: (18623, 7)
Products represented: 1057
Unique ingredients: 4664

First 5 rows:


,ingredient,semantic_labels
0,water,['mineral']
1,lychee juice from concentrate 25%,[]
2,coconut 25%,[]
3,fructose,"['sweetener', 'sugar', 'added_sugar', 'carbohy..."
4,sucrose,"['sweetener', 'sugar', 'added_sugar', 'carbohy..."


## 2. Prepare Training Data

Split into train/val/test using stratified multi-label splitting.

In [3]:
# Prepare feature matrix and labels
X = data['ingredient'].values
y = np.array(data['label_vector'].tolist())

print(f"Feature matrix: {X.shape}")
print(f"Label matrix: {y.shape}")
print(f"Positive label ratio: {y.sum() / y.size:.2%}")

# Stratified split
train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    random_state=SEED
)

num_unique_train = len(set(tuple(l) for l in train_labels))
print(f"\nSplit sizes:")
print(f"  Train: {len(train_texts)} ({num_unique_train} unique label combos)")
print(f"  Val:   {len(val_texts)}")
print(f"  Test:  {len(test_texts)}")

Feature matrix: (18623,)
Label matrix: (18623, 43)
Positive label ratio: 3.78%

Split sizes:
  Train: 14898 (254 unique label combos)
  Val:   1862
  Test:  1863


## 3. Compute Class Weights

Handle class imbalance using inverse frequency weighting (same approach as allergen model).

In [4]:
# Compute weights for each semantic category
pos_counts = np.array(train_labels).sum(axis=0)
class_weights = compute_class_weights(torch.tensor(train_labels, dtype=torch.float32))

print("Class weights (inverse frequency):")
for i, (cat, weight) in enumerate(zip(categories, class_weights.numpy())):
    w_str = f"{weight:.2f}"
    counts_str = f"{int(pos_counts[i])}"
    flag = " ← rare" if weight > 3 else ""
    print(f"  {cat:30s}  weight={w_str:>6s}  positive_samples={counts_str}{flag}")

Class weights (inverse frequency):
  food_additive                   weight=  0.45  positive_samples=2021
  preservative                    weight=  0.83  positive_samples=360
  flavor_enhancer                 weight=  0.81  positive_samples=406
  sweetener                       weight=  0.57  positive_samples=1179
  emulsifier                      weight=  0.89  positive_samples=279
  stabilizer                      weight=  0.88  positive_samples=289
  thickener                       weight=  0.76  positive_samples=489
  antioxidant                     weight=  0.82  positive_samples=382
  acidulant                       weight=  0.82  positive_samples=379
  colorant                        weight=  0.84  positive_samples=355
  fat_source                      weight=  0.55  positive_samples=1270
  oil_source                      weight=  0.65  positive_samples=819
  protein_source                  weight=  0.62  positive_samples=917
  carbohydrate_source             weight=  0.47  pos

## 4. Load Model

Initialize a fresh MobileBERT for semantic classification (separate from the allergen model).
The num_labels is set to the number of semantic categories.

In [5]:
MODEL_NAME = "google/mobilebert-uncased"
NUM_CATEGORIES = len(categories)
MAX_LENGTH = 32  # Individual ingredients are short
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CATEGORIES,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True,  # new classification head
)

# Move to available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded on {device}")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading google/mobilebert-uncased...


Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on cuda
  Layers: 24
  Parameters: 24,603,947


## 5. Training Setup

Uses the same weighted BCE loss and training loop as the allergen model.
Early stopping with patience=3 monitors validation macro F1.

In [6]:
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Tokenize all data
def tokenize_batch(texts):
    return tokenizer(
        texts.tolist() if hasattr(texts, 'tolist') else list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )

train_encodings = tokenize_batch(train_texts)
val_encodings = tokenize_batch(val_texts)

# Create DataLoaders
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    torch.tensor(train_labels, dtype=torch.float32),
)
val_dataset = TensorDataset(
    val_encodings['input_ids'],
    val_encodings['attention_mask'],
    torch.tensor(val_labels, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

print(f"Training setup:")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {int(0.1 * total_steps)}")

Training setup:
  Train samples: 14898
  Validation samples: 1862
  Batch size: 16
  Max length: 32
  Learning rate: 2e-05
  Epochs: 10
  Total steps: 9320
  Warmup steps: 932


## 6. Training Loop

Weighted BCE loss, early stopping by macro F1 on validation set.
Training code follows the same pattern as Notebook 07 (allergen training).

In [7]:
# === TRAINING LOOP ===

from sklearn.metrics import f1_score

loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights.to(device))

best_macro_f1 = 0.0
best_epoch = 0
patience_counter = 0

for epoch in range(EPOCHS):
    # Training
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = [x.to(device) for x in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    val_preds, val_probs = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = [x.to(device) for x in batch]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)
            val_probs.append(probs.cpu().numpy())
    
    val_probs = np.concatenate(val_probs)
    val_preds = (val_probs >= 0.5).astype(int)
    macro_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    avg_loss = total_loss / len(train_loader)
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={avg_loss:.4f}  val_macro_f1={macro_f1:.4f}")
    
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_epoch = epoch
        patience_counter = 0
        # Save best model
        model.save_pretrained("../models/mobilebert_semantic_final/")
        tokenizer.save_pretrained("../models/mobilebert_semantic_final/")
        print(f"  → New best model saved (macro_f1={macro_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= 3:
            print(f"  Early stopping at epoch {epoch+1}")
            break

Epoch  1/10  loss=359608.6723  val_macro_f1=0.6990
  → New best model saved (macro_f1=0.6990)
Epoch  2/10  loss=0.6056  val_macro_f1=0.8674
  → New best model saved (macro_f1=0.8674)
Epoch  3/10  loss=0.0542  val_macro_f1=0.8699
  → New best model saved (macro_f1=0.8699)
Epoch  4/10  loss=0.4709  val_macro_f1=0.8863
  → New best model saved (macro_f1=0.8863)
Epoch  5/10  loss=0.0082  val_macro_f1=0.8901
  → New best model saved (macro_f1=0.8901)
Epoch  6/10  loss=0.0293  val_macro_f1=0.8943
  → New best model saved (macro_f1=0.8943)
Epoch  7/10  loss=0.0218  val_macro_f1=0.8899
Epoch  8/10  loss=0.0196  val_macro_f1=0.8965
  → New best model saved (macro_f1=0.8965)
Epoch  9/10  loss=0.0020  val_macro_f1=0.8983
  → New best model saved (macro_f1=0.8983)
Epoch 10/10  loss=0.3846  val_macro_f1=0.8991
  → New best model saved (macro_f1=0.8991)


## 7. Threshold Optimization

Find optimal per-class probability thresholds (same approach as allergen model).

In [8]:
# Load best model and optimize thresholds
model_path = "../models/mobilebert_semantic_final/"
if os.path.exists(model_path):
    model, tokenizer, device = load_model_and_tokenizer(model_path)
    _, val_probs = predict_ml(val_texts, model, tokenizer, device,
                              thresholds=np.array([0.5] * NUM_CATEGORIES),
                              max_length=MAX_LENGTH)
    best_thresholds = find_best_thresholds(val_probs, np.array(val_labels))
    print("Optimized thresholds:")
    for i, cat in enumerate(categories):
        print(f"  {cat:30s}  threshold={best_thresholds[i]:.2f}")
else:
    print(f"Model not found at {model_path}. Train the model first in the cell above.")

Optimized thresholds:
  food_additive                   threshold=0.28
  preservative                    threshold=0.32
  flavor_enhancer                 threshold=0.24
  sweetener                       threshold=0.04
  emulsifier                      threshold=0.22
  stabilizer                      threshold=0.03
  thickener                       threshold=0.45
  antioxidant                     threshold=0.03
  acidulant                       threshold=0.07
  colorant                        threshold=0.08
  fat_source                      threshold=0.30
  oil_source                      threshold=0.09
  protein_source                  threshold=0.16
  carbohydrate_source             threshold=0.17
  animal_derived                  threshold=0.26
  plant_derived                   threshold=0.12
  milk_derivative                 threshold=0.94
  egg_derivative                  threshold=0.13
  soy_derivative                  threshold=0.12
  wheat_derivative                threshold=0.4

## 8. Evaluate on Test Set

In [9]:
model_path = "../models/mobilebert_semantic_final/"
if os.path.exists(model_path):
    test_preds, _ = predict_ml(test_texts, model, tokenizer, device, thresholds=best_thresholds, max_length=MAX_LENGTH)
    print_classification_report(test_labels, test_preds, categories, prefix="Semantic Classifier")
else:
    print(f"Model not found at {model_path}. Train and optimize thresholds first.")

=== Semantic Classifier ===
                     precision    recall  f1-score   support

      food_additive     0.9843    0.9921    0.9882       253
       preservative     1.0000    1.0000    1.0000        45
    flavor_enhancer     1.0000    0.9608    0.9800        51
          sweetener     0.9732    0.9864    0.9797       147
         emulsifier     0.9444    0.9714    0.9577        35
         stabilizer     0.9474    1.0000    0.9730        36
          thickener     0.9836    0.9836    0.9836        61
        antioxidant     0.9231    1.0000    0.9600        48
          acidulant     0.9592    0.9792    0.9691        48
           colorant     0.9000    1.0000    0.9474        45
         fat_source     0.9877    1.0000    0.9938       161
         oil_source     0.9722    0.9906    0.9813       106
     protein_source     0.9823    0.9652    0.9737       115
carbohydrate_source     1.0000    0.9959    0.9979       242
     animal_derived     0.9933    1.0000    0.9967      

## 9. Summary

In [10]:
print("=" * 50)
print("SEMANTIC CLASSIFICATION MODEL TRAINING")
print("=" * 50)
print(f"  Model: MobileBERT")
print(f"  Task: Multi-label semantic classification ({NUM_CATEGORIES} categories)")
print(f"  Training samples: {len(train_texts)}")
print(f"  Validation samples: {len(val_texts)}")
print(f"  Test samples: {len(test_texts)}")
print(f"  Max seq length: {MAX_LENGTH}")
print()
print("📋 Next step: Run 06_allergen_labeling.ipynb to continue allergen pipeline")

SEMANTIC CLASSIFICATION MODEL TRAINING
  Model: MobileBERT
  Task: Multi-label semantic classification (43 categories)
  Training samples: 14898
  Validation samples: 1862
  Test samples: 1863
  Max seq length: 32

📋 Next step: Run 06_allergen_labeling.ipynb to continue allergen pipeline
